In [0]:
%sql
CREATE OR REPLACE TEMP VIEW hco_org_details AS
WITH hco_npis AS (

    -- Medical (Elaprase NDC)
    SELECT DISTINCT
        BILLING_NPI                                AS HCO_NPI
        
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700') and  SERVICE_DATE BETWEEN '2023-08-01' AND '2025-11-30'

    UNION

    -- Pharmacy (Elaprase NDC | Paid only)
    SELECT DISTINCT
        
        PHARMACY_NPI                                AS HCO_NPI
        
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID' AND FILL_DATE BETWEEN '2023-08-01' AND '2025-11-30'

    UNION

    -- Medical (Elaprase Procedure codes)
    SELECT DISTINCT
        BILLING_NPI                                AS HCO_NPI
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    ) and  SERVICE_DATE BETWEEN '2023-08-01' AND '2025-11-30'
) 


SELECT DISTINCT
    n.HCO_NPI,
    p.ORGANIZATION_NAME,
    p.primary_specialty,
    p.secondary_specialty,
    p.PROVIDER_ADDRESS,
    p.PROVIDER_CITY,
    p.PROVIDER_STATE,
    p.PROVIDER_ZIP
FROM hco_npis n
JOIN com_edp_prd.com_raw.kom_providers p
  ON n.HCO_NPI = p.NPI
WHERE p.PROVIDER_TYPE = 'ORGANIZATION';


In [0]:
%sql
select DISTINCT PRIMARY_SPECIALTY, count(distinct hco_npi) from hco_org_details group by 1 order by 2 desc